# Notebook 04 — Carga, orquestación y buenas prácticas

Cuarto y último sub-bloque del Tema 04. Tomas los seis DataFrames del Notebook 03 y los **cargas** al `northwind_dwh_py` con `to_sql`. Después empaquetas todo el pipeline (extract + transform + load) en un script Python productivo con logging, manejo de errores y validaciones post-carga.

Cierra el tema con un panorama de cómo se ve un ETL **en producción real** — más allá del notebook.

Al terminar este notebook deberías tener el `northwind_dwh_py` poblado por tu ETL Python (equivalente al `northwind_dwh` que cargaste con scripts SQL en el Tema 01) y un `etl_pipeline.py` que puede correr de extremo a extremo.

**Contenido de este notebook:**

- [Setup](#setup)
- [Carga al DWH con `to_sql`](#carga-al-dwh-con-to_sql)
- [Estrategias: `replace`, `append`, `upsert`](#estrategias-replace-append-upsert)
- [Tipos correctos al cargar — `dtype`](#tipos-correctos-al-cargar--dtype)
- [Orden de carga — importa por las FKs](#orden-de-carga--importa-por-las-fks)
- [Pipeline modular — de notebook a `etl_pipeline.py`](#pipeline-modular--de-notebook-a-etl_pipelinepy)
- [Logging básico](#logging-básico)
- [Manejo de errores y rollback](#manejo-de-errores-y-rollback)
- [Validaciones post-carga](#validaciones-post-carga)
- [Cierre — ETL en producción real](#cierre--etl-en-producción-real)

## Setup

Re-creación del engine y re-ejecución condensada de los Notebooks 01-03 (extract + clean + transform) para tener los 6 DataFrames listos en memoria: `dim_customer`, `dim_product`, `dim_employee`, `dim_shipper`, `dim_date`, `fact_sales`.

Notebook auto-contenido: este bloque te deja en el estado final del Notebook 03.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

# === Extract (Notebook 01) ===
df_customers     = pd.read_sql("SELECT * FROM northwind_oltp.customers",     engine)
df_orders        = pd.read_sql("SELECT * FROM northwind_oltp.orders",        engine)
df_order_details = pd.read_sql("SELECT * FROM northwind_oltp.order_details", engine)
df_products      = pd.read_sql("SELECT * FROM northwind_oltp.products",      engine)
df_categories    = pd.read_sql("SELECT * FROM northwind_oltp.categories",    engine)
df_suppliers     = pd.read_sql("SELECT * FROM northwind_oltp.suppliers",     engine)
df_employees     = pd.read_sql("SELECT * FROM northwind_oltp.employees",     engine)
df_shippers      = pd.read_sql("SELECT * FROM northwind_oltp.shippers",      engine)

# === Clean (Notebook 02) ===
df_orders["order_date"]    = pd.to_datetime(df_orders["order_date"],    errors="coerce")
df_orders["required_date"] = pd.to_datetime(df_orders["required_date"], errors="coerce")
df_orders["shipped_date"]  = pd.to_datetime(df_orders["shipped_date"],  errors="coerce")
df_order_details["unit_price"] = df_order_details["unit_price"].round(2)
df_order_details["discount"]   = df_order_details["discount"].round(2)
df_order_details["extended_price"] = (df_order_details["quantity"] * df_order_details["unit_price"]).round(2)
df_order_details["line_total"]     = (
    df_order_details["quantity"] * df_order_details["unit_price"] * (1 - df_order_details["discount"])
).round(2)

# === Transform (Notebook 03) ===

# dim_customer
dim_customer = df_customers[[
    "customer_id", "company_name", "contact_name", "contact_title",
    "city", "region", "postal_code", "country"
]].reset_index(drop=True)
dim_customer.insert(0, "customer_key", dim_customer.index + 1)

# dim_product
dim_product = (
    df_products[["product_id", "product_name", "category_id", "supplier_id", "discontinued"]]
    .merge(df_categories[["category_id", "category_name", "description"]]
             .rename(columns={"description": "category_desc"}),
           on="category_id", how="inner")
    .merge(df_suppliers[["supplier_id", "company_name", "country", "city"]]
             .rename(columns={"company_name": "supplier_name", "country": "supplier_country", "city": "supplier_city"}),
           on="supplier_id", how="inner")
)
dim_product["discontinued"] = dim_product["discontinued"].astype(bool)
dim_product = dim_product[[
    "product_id", "product_name", "category_id", "category_name", "category_desc",
    "supplier_id", "supplier_name", "supplier_country", "supplier_city", "discontinued"
]].reset_index(drop=True)
dim_product.insert(0, "product_key", dim_product.index + 1)

# dim_employee
empleado = df_employees[[
    "employee_id", "first_name", "last_name", "title", "city",
    "country", "region", "hire_date", "reports_to"
]].copy()
empleado["full_name"] = empleado["first_name"] + " " + empleado["last_name"]
manager = df_employees[["employee_id", "first_name", "last_name"]].copy()
manager["reports_to_name"] = manager["first_name"] + " " + manager["last_name"]
manager = manager[["employee_id", "reports_to_name"]].rename(columns={"employee_id": "manager_id"})
dim_employee = empleado.merge(manager, left_on="reports_to", right_on="manager_id", how="left")
dim_employee = dim_employee[[
    "employee_id", "full_name", "title", "city", "country", "region", "hire_date", "reports_to_name"
]].reset_index(drop=True)
dim_employee.insert(0, "employee_key", dim_employee.index + 1)

# dim_shipper
dim_shipper = df_shippers[["shipper_id", "company_name", "phone"]].reset_index(drop=True)
dim_shipper.insert(0, "shipper_key", dim_shipper.index + 1)

# dim_date
fechas = pd.date_range(start="1996-01-01", end="1998-12-31", freq="D")
dim_date = pd.DataFrame({"full_date": fechas})
dim_date["date_key"]           = dim_date["full_date"].dt.year * 10000 + dim_date["full_date"].dt.month * 100 + dim_date["full_date"].dt.day
dim_date["year"]               = dim_date["full_date"].dt.year
dim_date["quarter"]            = dim_date["full_date"].dt.quarter
dim_date["month_number"]       = dim_date["full_date"].dt.month
dim_date["week_of_year"]       = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["day_of_month"]       = dim_date["full_date"].dt.day
dim_date["day_of_week_number"] = dim_date["full_date"].dt.dayofweek + 1
dim_date["is_weekend"]         = dim_date["day_of_week_number"].isin([6, 7])
meses = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
         "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
dias  = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
dim_date["month_name"]       = dim_date["month_number"].map(lambda m: meses[m-1])
dim_date["day_of_week_name"] = dim_date["day_of_week_number"].map(lambda d: dias[d-1])
dim_date = dim_date[[
    "date_key", "full_date", "year", "quarter", "month_number", "month_name",
    "week_of_year", "day_of_month", "day_of_week_number", "day_of_week_name", "is_weekend"
]]

# fact_sales
fact = df_order_details[[
    "order_id", "product_id", "quantity", "unit_price", "discount", "extended_price", "line_total"
]].copy()
fact = fact.merge(df_orders[[
    "order_id", "customer_id", "employee_id", "ship_via",
    "order_date", "required_date", "shipped_date"
]], on="order_id", how="inner")
fact = fact.merge(dim_customer[["customer_id", "customer_key"]], on="customer_id", how="inner")
fact = fact.merge(dim_product[["product_id", "product_key"]],    on="product_id",  how="inner")
fact = fact.merge(dim_employee[["employee_id", "employee_key"]], on="employee_id", how="inner")
fact = fact.merge(dim_shipper[["shipper_id", "shipper_key"]],    left_on="ship_via", right_on="shipper_id", how="left")

def to_date_key(series):
    return (series.dt.year * 10000 + series.dt.month * 100 + series.dt.day).astype("Int64")

fact["order_date_key"]    = to_date_key(fact["order_date"])
fact["required_date_key"] = to_date_key(fact["required_date"])
fact["shipped_date_key"]  = to_date_key(fact["shipped_date"])

fact_sales = fact[[
    "order_id", "customer_key", "product_key", "employee_key", "shipper_key",
    "order_date_key", "required_date_key", "shipped_date_key",
    "quantity", "unit_price", "discount", "extended_price", "line_total",
]].reset_index(drop=True)
fact_sales.insert(0, "sale_key", fact_sales.index + 1)

print("DataFrames listos para cargar:")
for nombre, df in [
    ("dim_customer", dim_customer), ("dim_product", dim_product),
    ("dim_employee", dim_employee), ("dim_shipper", dim_shipper),
    ("dim_date",     dim_date),     ("fact_sales",  fact_sales),
]:
    print(f"  {nombre:14s} {df.shape}")

### Schema de destino: `northwind_dwh_py`

Vas a cargar a un schema **separado** del `northwind_dwh` que poblaste con SQL en el Tema 01. Dos razones:

1. No destruyes el trabajo anterior — el `northwind_dwh` cargado por SQL queda intacto para comparar.
2. `to_sql` con `if_exists='replace'` haría `DROP TABLE` — perderías las `GENERATED ALWAYS AS IDENTITY` y las `GENERATED ALWAYS AS … STORED` que definiste con cuidado en el DDL.

Al final podrás correr la misma query analítica contra ambos schemas y verificar que dan el mismo resultado: dos caminos (SQL puro y Python) llegando al mismo destino.

In [ ]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS northwind_dwh_py"))

print("✓ schema northwind_dwh_py listo")

## Carga al DWH con `to_sql`

El método principal para escribir un DataFrame a una tabla SQL es `df.to_sql`. Sintaxis:

```python
df.to_sql(
    name='dim_customer',          # nombre de la tabla destino
    con=engine,                   # engine de SQLAlchemy
    schema='northwind_dwh_py',    # opcional, para schemas distintos del default
    if_exists='replace',          # 'fail' | 'replace' | 'append'
    index=False,                  # NO escribir el índice de pandas como columna
    chunksize=1000,               # insertar en lotes de 1000 filas
    method='multi',               # múltiples filas por INSERT (más rápido)
)
```

**Parámetros clave a entender:**

- **`index=False`** — casi siempre lo quieres. El índice de pandas (0..N-1) es un detalle interno; no debería viajar al DWH como columna.
- **`chunksize`** — controla cuántas filas se mandan por viaje a la red. `None` (default) manda todo en un INSERT enorme; valores típicos en producción van de 500 a 5 000.
- **`method='multi'`** — agrupa varias filas en un solo `INSERT VALUES (…), (…), (…)` en vez de un INSERT por fila. Mejora el throughput entre 10× y 50× con PostgreSQL.

In [ ]:
# Primera carga: dim_customer al schema nuevo
dim_customer.to_sql(
    name="dim_customer",
    con=engine,
    schema="northwind_dwh_py",
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi",
)

# Verificar contra el origen
n = pd.read_sql("SELECT count(*) AS n FROM northwind_dwh_py.dim_customer", engine)["n"][0]
print(f"✓ dim_customer cargada: {n} filas (esperadas: {len(dim_customer)})")

## Estrategias: `replace`, `append`, `upsert`

El parámetro `if_exists` decide qué pasa cuando la tabla destino **ya existe**. Tres opciones:

| Valor | Comportamiento | Cuándo usarlo |
|---|---|---|
| `'fail'` | Lanza excepción si la tabla existe | Para detectar cargas accidentales que sobreescribirían datos |
| `'replace'` | `DROP TABLE` + `CREATE TABLE` + `INSERT` | Reload completo del DWH. **Idempotente** — correr 2 o 20 veces produce el mismo resultado |
| `'append'` | Solo `INSERT` | Cargas **incrementales** (solo las filas nuevas del día) |

**Trampa importante con `'append'`:** no es idempotente. Si reejecutas el ETL sin filtrar las filas que ya cargaste, **duplicas datos**. Combinaciones típicas:

- `replace` para tablas pequeñas (las 5 dimensiones de Northwind caben en milisegundos).
- `append` para fact tables grandes, **filtrando por ventana de tiempo** ("solo ventas con `order_date >= ayer`").
- **`upsert`** (INSERT … ON CONFLICT UPDATE) cuando una fila puede aparecer una vez y luego actualizarse — pandas no lo soporta de fábrica, hay que escribir SQL crudo.

**¿Qué hace un upsert exactamente?**

Imagina que tu carga incremental del día trae los 3 primeros clientes de Northwind, pero `ALFKI` ya está en `dim_customer` con datos viejos. Quieres: **insertar `ANATR` y `ANTON` que son nuevos, y actualizar `ALFKI` con los valores nuevos**. En una sola query:

```sql
INSERT INTO northwind_dwh_py.dim_customer
    (customer_key, customer_id, company_name, city, country)
VALUES
    (1, 'ALFKI', 'Alfreds Futterkiste',      'Berlin',      'Germany'),
    (2, 'ANATR', 'Ana Trujillo Emparedados', 'México D.F.', 'Mexico'),
    (3, 'ANTON', 'Antonio Moreno Taquería',  'México D.F.', 'Mexico')
ON CONFLICT (customer_key) DO UPDATE SET
    company_name = EXCLUDED.company_name,
    city         = EXCLUDED.city,
    country      = EXCLUDED.country;
```

Resultado:

- `ALFKI` ya existía (`customer_key = 1`) → **se actualiza** con los nuevos valores.
- `ANATR` y `ANTON` no existían → **se insertan**.
- Una sola query, una sola transacción, sin chequeo previo.

In [ ]:
# Upsert manual con SQLAlchemy — patrón productivo, no lo trae pandas
from sqlalchemy import text

upsert_sql = text("""
    INSERT INTO northwind_dwh_py.dim_customer
        (customer_key, customer_id, company_name, contact_name, contact_title,
         city, region, postal_code, country)
    VALUES
        (:customer_key, :customer_id, :company_name, :contact_name, :contact_title,
         :city, :region, :postal_code, :country)
    ON CONFLICT (customer_key) DO UPDATE SET
        company_name  = EXCLUDED.company_name,
        contact_name  = EXCLUDED.contact_name,
        contact_title = EXCLUDED.contact_title,
        city          = EXCLUDED.city,
        region        = EXCLUDED.region,
        postal_code   = EXCLUDED.postal_code,
        country       = EXCLUDED.country
""")

# Para que ON CONFLICT funcione, customer_key debe ser PRIMARY KEY o UNIQUE.
# Aquí no lo es (pandas creó la tabla sin PK), así que solo mostramos el patrón.
# En el script productivo crearías la tabla con su DDL completo primero.
print("Patrón de upsert documentado. EXCLUDED.* refiere a los valores que SE INTENTARON insertar.")

El uso de **`EXCLUDED`** es clave: dentro del `DO UPDATE`, `EXCLUDED.col` representa el valor que SE INTENTÓ insertar (vino del DataFrame); `col` a secas representa el valor actual en la tabla. Eso te permite distinguir y combinar ambos lados.

## Tipos correctos al cargar — `dtype`

Sin `dtype` explícito, pandas + SQLAlchemy infieren los tipos SQL. La inferencia suele caer en:

- `BIGINT` para cualquier entero (aunque sea un `SMALLINT` chiquito).
- `DOUBLE PRECISION` (= `float8`) para decimales. **Eso revive el problema de REAL/FLOAT** que viste en el Tema 02: imprecisión binaria en columnas de dinero.
- `TEXT` para strings (no `VARCHAR(n)`).

**Patrón:** pasa un dict `dtype={"col": SQLAlchemyType}` mapeando las columnas críticas a sus tipos correctos.

```python
from sqlalchemy.types import Integer, Numeric, String, Date, Boolean, SmallInteger
```

In [ ]:
from sqlalchemy.types import Integer, Numeric, String, Date, Boolean, SmallInteger

fact_sales_dtypes = {
    "sale_key":           Integer(),
    "order_id":           SmallInteger(),
    "customer_key":       Integer(),
    "product_key":        Integer(),
    "employee_key":       Integer(),
    "shipper_key":        Integer(),
    "order_date_key":     Integer(),
    "required_date_key":  Integer(),
    "shipped_date_key":   Integer(),
    "quantity":           SmallInteger(),
    "unit_price":         Numeric(10, 2),    # ← dinero como decimal exacto
    "discount":           Numeric(4, 2),
    "extended_price":     Numeric(12, 2),
    "line_total":         Numeric(12, 2),
}

fact_sales.to_sql(
    name="fact_sales",
    con=engine,
    schema="northwind_dwh_py",
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi",
    dtype=fact_sales_dtypes,
)

# Verificar tipos en la tabla creada
pd.read_sql("""
    SELECT column_name, data_type, numeric_precision, numeric_scale
    FROM information_schema.columns
    WHERE table_schema = 'northwind_dwh_py' AND table_name = 'fact_sales'
    ORDER BY ordinal_position
""", engine)

## Orden de carga — importa por las FKs

Si tu DWH tiene constraints `FOREIGN KEY` declaradas, **carga las dimensiones primero y la fact al final**. La fact apunta a las dims; cargarla antes haría que cada `INSERT` falle por FK no encontrada.

Orden recomendado:

```
1. dim_date          (independiente)
2. dim_customer      (independiente)
3. dim_product       (independiente)
4. dim_employee      (independiente)
5. dim_shipper       (independiente)
6. fact_sales        (apunta a las 5 anteriores)
```

**Nota sobre este notebook:** como estamos usando `if_exists='replace'`, pandas crea las tablas **sin** FKs declaradas — el orden no falla técnicamente. Pero seguimos respetándolo porque (a) refleja la lógica de dependencias, (b) en producción con DDL real sí importaría.

In [ ]:
# Carga ordenada de las 5 dims + fact_sales
tablas_en_orden = [
    ("dim_date",     dim_date),
    ("dim_customer", dim_customer),
    ("dim_product",  dim_product),
    ("dim_employee", dim_employee),
    ("dim_shipper",  dim_shipper),
    ("fact_sales",   fact_sales),
]

for nombre, df in tablas_en_orden:
    df.to_sql(
        name=nombre,
        con=engine,
        schema="northwind_dwh_py",
        if_exists="replace",
        index=False,
        chunksize=1000,
        method="multi",
    )
    print(f"✓ {nombre:14s} {len(df):>5} filas")

print("\nCarga completa.")

## Pipeline modular — de notebook a `etl_pipeline.py`

El notebook es para **explorar y aprender**. Un ETL productivo vive en un **script `.py`** que se ejecuta desde la línea de comandos. Diferencias clave:

| Notebook | Script productivo |
|---|---|
| Estado entre celdas | Sin estado: todo arranca desde cero |
| Ejecución interactiva | Determinístico, automatizable |
| Variables sueltas | Funciones con argumentos explícitos |
| Configuración hardcoded | Vía CLI args o variables de entorno |
| `print` para inspeccionar | `logging` con niveles y timestamp |

**Estructura mínima del script:**

- `extract(engine) -> dict[str, DataFrame]` — todas las queries de extracción en una sola función.
- `transform(raw_data) -> dict[str, DataFrame]` — devuelve los 6 DataFrames del modelo dimensional.
- `load(transformed, engine)` — carga ordenada al DWH.
- `validate(engine)` — verificaciones post-carga.
- `main()` — orquesta todo, parsea argumentos, configura logging.

In [ ]:
# Esqueleto de etl_pipeline.py — guárdalo como archivo aparte y ejecútalo con:
#   python etl_pipeline.py --host ... --password ... --database northwind

esqueleto = '''\
import argparse
import logging
import pandas as pd
from sqlalchemy import create_engine, text

logger = logging.getLogger(__name__)


def extract(engine) -> dict:
    """Extrae las 8 tablas del OLTP en DataFrames."""
    logger.info("Extracción: leyendo OLTP")
    return {
        "customers":     pd.read_sql("SELECT * FROM northwind_oltp.customers",     engine),
        "orders":        pd.read_sql("SELECT * FROM northwind_oltp.orders",        engine),
        "order_details": pd.read_sql("SELECT * FROM northwind_oltp.order_details", engine),
        "products":      pd.read_sql("SELECT * FROM northwind_oltp.products",      engine),
        "categories":    pd.read_sql("SELECT * FROM northwind_oltp.categories",    engine),
        "suppliers":     pd.read_sql("SELECT * FROM northwind_oltp.suppliers",     engine),
        "employees":     pd.read_sql("SELECT * FROM northwind_oltp.employees",     engine),
        "shippers":      pd.read_sql("SELECT * FROM northwind_oltp.shippers",      engine),
    }


def transform(raw: dict) -> dict:
    """Construye las 5 dims + fact_sales en memoria."""
    logger.info("Transformación: construyendo modelo dimensional")
    # … lógica del Notebook 03: limpieza + dim_* + fact_sales …
    return {"dim_customer": ..., "fact_sales": ..., ...}


def load(transformed: dict, engine, schema: str = "northwind_dwh_py"):
    """Carga ordenada al DWH (dims primero, fact al final)."""
    logger.info("Carga: escribiendo al DWH")
    with engine.begin() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema}"))

    orden = ["dim_date", "dim_customer", "dim_product",
             "dim_employee", "dim_shipper", "fact_sales"]
    for nombre in orden:
        df = transformed[nombre]
        df.to_sql(nombre, engine, schema=schema, if_exists="replace",
                  index=False, chunksize=1000, method="multi")
        logger.info("  %s: %d filas", nombre, len(df))


def validate(engine, schema: str = "northwind_dwh_py"):
    """Validaciones post-carga: conteos y agregados contra el OLTP."""
    logger.info("Validación: comparando DWH vs OLTP")
    # … queries de validación …


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--host",     required=True)
    parser.add_argument("--password", required=True)
    parser.add_argument("--database", default="northwind")
    args = parser.parse_args()

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    )

    engine = create_engine(
        f"postgresql+psycopg2://postgres:{args.password}@{args.host}:5432/{args.database}"
    )

    try:
        raw         = extract(engine)
        transformed = transform(raw)
        load(transformed, engine)
        validate(engine)
        logger.info("ETL completado correctamente")
    except Exception as e:
        logger.exception("ETL falló: %s", e)
        raise


if __name__ == "__main__":
    main()
'''

print(esqueleto)

## Logging básico

**Regla operativa: en producción, `print` está prohibido.** Usa el módulo `logging` de la stdlib.

Razones técnicas:

- **Niveles** (`DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL`) — puedes silenciar lo irrelevante en producción y subirlo en debug, sin tocar el código.
- **Formato configurable** — timestamp + nivel + módulo + mensaje, vital cuando lees un log de hace 3 días.
- **Múltiples destinos** — el mismo log puede ir a stdout, a un archivo, y a un servicio de observabilidad (CloudWatch, Datadog) al mismo tiempo.
- **Excepciones con stacktrace** — `logger.exception()` registra el traceback completo automáticamente.

Por qué importa: cuando el ETL corre a las 3am sin tu supervisión, el log es **lo único** que tienes para diagnosticar qué pasó.

In [ ]:
import logging

# Configuración global — una vez al inicio del script
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("etl")

# Niveles en uso
logger.debug("detalle fino — solo en debug")               # no se ve (level=INFO)
logger.info("paso normal del flujo — extracción iniciada")
logger.warning("algo raro pero recuperable — 3 filas con NULL en country")
logger.error("falla esperada y manejada — reintentando conexión")
logger.critical("falla irrecuperable — abortando ETL")

# Excepciones — incluye el traceback completo automáticamente
try:
    1 / 0
except ZeroDivisionError:
    logger.exception("división por cero al calcular tasa de descuento")

## Manejo de errores y rollback

En un ETL real las cosas fallan: la red se cae, un campo trae caracteres inesperados, una FK no existe. Necesitas dos defensas:

1. **Transacciones SQLAlchemy** — agrupar varias operaciones para que sean "todo o nada". Si una falla, las anteriores se deshacen automáticamente.
2. **`try/except`** alrededor de los pasos críticos, con log de la excepción.

**Patrón `engine.begin()`:** abre una transacción explícita. Si el bloque sale con excepción, `ROLLBACK`. Si sale normal, `COMMIT`. Sin tener que llamarlos a mano.

**Decisión clave: ¿reintentar o fallar rápido?** Errores transitorios (red, timeouts) ameritan retry. Errores de lógica (FK violada, dato malformado) deben fallar rápido — reintentar no va a arreglarlos.

In [ ]:
# Bloque transaccional con engine.begin() y manejo de excepciones

try:
    with engine.begin() as conn:
        # Si CUALQUIER operación dentro de este bloque falla, las anteriores
        # se hacen ROLLBACK automáticamente — el DWH queda como estaba.
        conn.execute(text("DELETE FROM northwind_dwh_py.fact_sales WHERE order_id < 0"))
        conn.execute(text("UPDATE northwind_dwh_py.dim_customer SET region = 'N/A' WHERE region IS NULL"))
        # Si aquí lanzo una excepción manualmente, las dos operaciones anteriores se revierten
        # raise RuntimeError("simulando un fallo")
    logger.info("Transacción confirmada (COMMIT automático al salir del with)")

except Exception as e:
    # logger.exception incluye el stacktrace; útil para diagnóstico
    logger.exception("Transacción revertida por excepción: %s", e)

**Retry conceptual (sin librería externa):**

```python
import time

for intento in range(1, 4):    # 3 intentos
    try:
        ejecutar_paso_critico()
        break
    except TransientError as e:
        logger.warning("intento %d falló: %s", intento, e)
        time.sleep(2 ** intento)    # backoff exponencial: 2s, 4s, 8s
else:
    raise RuntimeError("agotados los 3 intentos")
```

En producción real, librerías como [`tenacity`](https://tenacity.readthedocs.io) hacen esto declarativo con decoradores.

## Validaciones post-carga

Después de cargar, **verificar contra el origen**. Si alguna validación falla, el pipeline debería abortar — un DWH inconsistente es peor que ningún DWH (la gente confiará en él y tomará decisiones malas).

Tres niveles de validación:

1. **Conteos** — el grano debe coincidir: `count(*) DWH == count(*) OLTP`.
2. **Sumas y agregados** — totales monetarios y de unidades deben coincidir (con la tolerancia esperada por el redondeo a NUMERIC).
3. **Spot-checks** — una fila específica del OLTP debe poder ubicarse en el DWH con todas sus FKs resueltas correctamente.

In [ ]:
# Validación 1 — conteos coinciden con el origen
validaciones = pd.read_sql("""
    SELECT 'fact_sales'   AS tabla,
           (SELECT count(*) FROM northwind_oltp.order_details) AS oltp,
           (SELECT count(*) FROM northwind_dwh_py.fact_sales)  AS dwh
    UNION ALL
    SELECT 'dim_customer',
           (SELECT count(*) FROM northwind_oltp.customers),
           (SELECT count(*) FROM northwind_dwh_py.dim_customer)
    UNION ALL
    SELECT 'dim_product',
           (SELECT count(*) FROM northwind_oltp.products),
           (SELECT count(*) FROM northwind_dwh_py.dim_product)
    UNION ALL
    SELECT 'dim_employee',
           (SELECT count(*) FROM northwind_oltp.employees),
           (SELECT count(*) FROM northwind_dwh_py.dim_employee)
    UNION ALL
    SELECT 'dim_shipper',
           (SELECT count(*) FROM northwind_oltp.shippers),
           (SELECT count(*) FROM northwind_dwh_py.dim_shipper);
""", engine)

validaciones["diff"] = validaciones["oltp"] - validaciones["dwh"]
print(validaciones)

assert (validaciones["diff"] == 0).all(), "conteos no coinciden — abortar pipeline"
print("\n✓ Todos los conteos coinciden")

In [ ]:
# Validación 2 — SUM(line_total) y SUM(quantity) cuadran
totales = pd.read_sql("""
    SELECT
        (SELECT SUM(quantity) FROM northwind_oltp.order_details)  AS qty_oltp,
        (SELECT SUM(quantity) FROM northwind_dwh_py.fact_sales)   AS qty_dwh,
        ROUND((SELECT SUM(quantity * unit_price * (1 - discount))::NUMERIC
               FROM northwind_oltp.order_details), 2)             AS net_oltp,
        (SELECT SUM(line_total) FROM northwind_dwh_py.fact_sales) AS net_dwh
""", engine)

print(totales.T)
print(f"\n  Diferencia de cantidades:    {totales['qty_dwh'][0] - totales['qty_oltp'][0]}")
print(f"  Diferencia de ventas netas:  {float(totales['net_dwh'][0]) - float(totales['net_oltp'][0]):.4f}")
print("  (centavos de diferencia son esperados por el redondeo de REAL→NUMERIC)")

In [ ]:
# Validación 3 — spot check: ventas netas por categoría en 1997
# Misma query que ejecutaste contra northwind_dwh en el Tema 02. Debe dar el mismo resultado.
ventas_por_categoria = pd.read_sql("""
    SELECT dp.category_name,
           COUNT(*)                AS lineas,
           ROUND(SUM(fs.line_total), 2) AS ventas_netas
    FROM      northwind_dwh_py.fact_sales fs
    JOIN      northwind_dwh_py.dim_product dp USING (product_key)
    JOIN      northwind_dwh_py.dim_date    dd ON dd.date_key = fs.order_date_key
    WHERE     dd.year = 1997
    GROUP BY  dp.category_name
    ORDER BY  ventas_netas DESC;
""", engine)

print(ventas_por_categoria)

## Cierre — ETL en producción real

Lo que hiciste en estos cuatro notebooks es la **lógica fundamental** del ETL: extraer, perfilar, limpiar, transformar al modelo analítico, cargar y validar. En la industria, esa lógica se monta sobre infraestructura más sofisticada — pero **los conceptos no cambian**.

**Object storage (S3) como zona de aterrizaje** — los archivos del día llegan a S3 (CSV de socios comerciales, exports nocturnos de sistemas operacionales, eventos en streaming). El ETL los lee desde ahí. Ventaja: barato, separa storage de compute, queda como auditoría de la fuente original.

**Formatos columnares (Parquet, ORC)** — muchísimo más eficientes que CSV para datasets analíticos. Comprimen 5-10× mejor, leen solo las columnas que necesitas, traen esquema y tipos embebidos. Si tu OLTP no produce Parquet, conviértelo en la primera etapa del ETL.

**Orquestadores (Airflow, Prefect, Dagster)** — el script `etl_pipeline.py` que viste corre una vez, manual. En producción quieres: scheduling (`@daily a las 02:00`), dependencias entre tareas (DAG), reintentos automáticos, alertas si algo falla, UI para ver el historial de runs. Esos orquestadores son el siguiente nivel de abstracción.

**Compute serverless (AWS Glue, Lambda)** — el ETL como función que escala sin servidores que mantener. Pagas solo el tiempo de cómputo que usaste. Útil para cargas espasmódicas.

**Tendencia ELT moderna (dbt + Snowflake / BigQuery / Redshift)** — invertir el orden: Extract → Load (cargar crudo al DWH columnar) → Transform con SQL **dentro** del DWH. Razón: los DWHs modernos son tan rápidos que transformar ahí sale mejor que en Python. dbt es la herramienta dominante en este flujo; define transformaciones como modelos SQL versionados.

**Lo que sigue valiendo:** la decisión de qué constituye un buen modelo dimensional, cómo distinguir hechos de atributos, cómo evitar fan traps en joins, qué validaciones aplicar post-carga. Esas decisiones no las resuelve la herramienta; las resuelves tú.

## Y con esto cierras el Tema 04

**Lo que construiste en los cuatro notebooks:**

- **Notebook 01** — extracción desde Aurora con SQLAlchemy + `pd.read_sql`; lectura de CSV de Airbnb.
- **Notebook 02** — perfilado (`info`, `describe`, `isna`, `value_counts`) y limpieza (nulos, duplicados, normalización de strings, conversión de tipos, catálogos controlados).
- **Notebook 03** — transformación al modelo dimensional: 5 dimensiones desnormalizadas + `dim_date` sintética + `fact_sales` con merges sucesivos para resolver surrogate keys.
- **Notebook 04** — carga al DWH con `to_sql`, estrategias `replace`/`append`/`upsert`, pipeline modular en `etl_pipeline.py`, logging, manejo transaccional de errores y validaciones post-carga.

Ahora tu DWH (`northwind_dwh_py`) está cargado por **tu propio ETL Python**, equivalente al `northwind_dwh` que se pobló vía scripts SQL en el Tema 01. Puedes correr la misma query analítica contra ambos schemas y verificar que dan el mismo resultado.

**Lo que viene en el Tema 05 (SQL avanzado):** ahora que el DWH está poblado, las queries analíticas son el siguiente nivel. Saldrás del SELECT-FROM-JOIN-GROUP-BY básico hacia funciones más expresivas: `CASE`, agregados condicionales, manejo fino de NULLs, funciones de string y fecha de PostgreSQL.

---

<p align="center">
<a href="03_transformacion.ipynb">← Anterior: Notebook 03</a> | <a href="Readme.md">Volver al índice del Tema 04</a> | <a href="../Tema-05/Readme.md">Siguiente: Tema 05 →</a>
</p>